In [2]:
import os
import requests
import pandas as pd
import geopandas as gpd
import numpy as np
import pickle
from loguru import logger

from app.utils.config import DATA_PATH

from popframe.preprocessing.level_filler import LevelFiller
from popframe.models.region import Region
from idu_clients import UrbanAPI

URBAN_API = 'http://10.32.1.107:5300'
POPULATION_COUNT_INDICATOR_ID = 1
DEFAULT_CRS = 4326

In [3]:
async def load_region_bounds(region_id: int = None) -> gpd.GeoDataFrame:
    urban_api = UrbanAPI('http://10.32.1.107:5300')
    regions = await urban_api.get_regions()
    if regions.empty:
        raise FileNotFoundError(f"Region bounds for {region_id} not found.")
    return regions

def get_territories(parent_id : int | None = None, all_levels = False, geometry : bool = False) -> pd.DataFrame | gpd.GeoDataFrame:
    res = requests.get(URBAN_API + f'/api/v1/all_territories{"" if geometry else "_without_geometry"}', {
        'parent_id': parent_id,
        'get_all_levels': all_levels
    })
    res_json = res.json()
    if geometry:
        gdf = gpd.GeoDataFrame.from_features(res_json, crs=DEFAULT_CRS)
        return gdf.set_index('territory_id', drop=True)
    df = pd.DataFrame(res_json)
    return df.set_index('territory_id', drop=True)

def get_territories_population(territories_gdf : gpd.GeoDataFrame):
    res = requests.get(f'{URBAN_API}/api/v1/indicator/{POPULATION_COUNT_INDICATOR_ID}/values')
    res_df = pd.DataFrame(res.json())
    res_df = res_df[res_df['territory_id'].isin(territories_gdf.index)]
    res_df = res_df.groupby('territory_id').agg({'value': 'last'}).rename(columns={'value':'population'})
    return territories_gdf[['geometry', 'name']].merge(res_df, left_index=True, right_index=True)

def load_towns(region_id: int) -> gpd.GeoDataFrame:
    territories_gdf = get_territories(region_id, all_levels = True, geometry=True)
    territories_gdf['was_point'] = territories_gdf['properties'].apply(lambda p : p['was_point'] if 'was_point' in p else False)
    towns_gdf = territories_gdf[territories_gdf['was_point']]
    towns_gdf['geometry'] = towns_gdf['geometry'].representative_point()
    towns_gdf = get_territories_population(towns_gdf)
    towns_gdf['id'] = towns_gdf.index
    level_filler = LevelFiller(towns=towns_gdf)
    towns = level_filler.fill_levels()
    return towns

In [3]:
region_id = 1
regions = await load_region_bounds(region_id) if region_id is not None else await load_region_bounds()
region = regions.loc[[region_id]]
local_crs = region.geometry.estimate_utm_crs()
logger.info(f"Creating model for {region_id}...")

towns = load_towns(region_id)
towns

2024-11-02 15:23:48.577 | INFO     | __main__:<module>:5 - Creating model for 1...
/Users/mvin/Code/PopFrame_API/.venv/lib/python3.10/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


,geometry,id,name,population,level
territory_id,,,,,
207,POINT (33.75892 59.36226),207,Болото,10,Малое сельское поселение
208,POINT (33.786 59.47517),208,Большой Остров,68,Малое сельское поселение
209,POINT (33.79236 59.47356),209,Бор,1734,Большое сельское поселение
210,POINT (33.77572 59.44249),210,Бороватое,10,Малое сельское поселение
211,POINT (33.67728 59.32819),211,Бочево,10,Малое сельское поселение
...,...,...,...,...,...
3133,POINT (31.23421 59.17021),3133,Апраксин Бор,313,Среднее сельское поселение
3134,POINT (31.31924 59.18702),3134,Александровка,313,Среднее сельское поселение
3135,POINT (31.47462 59.29408),3135,Большая Горка,313,Среднее сельское поселение


In [4]:
res = requests.get('http://10.32.1.65:5700' + f'/api_v1/{1}/get_matrix', {
    'graph_type': 'car'
})
json = res.json()
adj_mx = pd.DataFrame(json['values'], index=json['index'], columns=json['columns'])




In [5]:
adj_mx

,207,208,209,210,211,212,213,214,215,216,...,3128,3129,3130,3131,3132,3133,3134,3135,3136,3137
207,0.000,11.687,12.024,10.794,7.768,3.431,21.873,29.029,8.415,17.122,...,175.052,167.133,165.000,170.273,195.276,195.276,188.976,172.755,171.412,183.300
208,11.687,0.000,0.337,4.313,18.027,13.690,29.914,37.070,16.456,10.347,...,163.986,156.067,153.934,159.207,184.210,184.210,177.910,161.689,160.346,172.234
209,12.024,0.337,0.000,4.650,18.364,14.027,30.251,37.407,16.248,10.684,...,163.649,155.730,153.597,158.870,183.873,183.873,177.573,161.352,160.009,171.897
210,10.794,4.313,4.650,0.000,17.134,12.797,29.021,36.177,15.563,9.748,...,167.678,159.759,157.626,162.899,187.902,187.902,181.602,165.381,164.038,175.926
211,7.768,18.027,18.364,17.134,0.000,4.337,28.213,35.369,14.755,23.462,...,181.392,173.473,171.340,176.613,201.616,201.616,195.316,179.095,177.752,189.640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3133,195.417,184.351,184.014,188.043,201.757,197.420,213.644,220.800,197.759,194.077,...,24.994,29.115,31.820,32.255,0.000,0.000,17.446,39.575,38.232,32.092
3134,189.116,178.050,177.713,181.742,195.456,191.119,207.343,214.499,191.458,187.776,...,17.942,22.063,24.768,25.203,17.446,17.446,0.000,32.523,31.180,25.040
3135,172.899,161.833,161.496,165.525,179.239,174.902,191.126,198.282,175.241,171.559,...,18.603,10.684,7.755,13.824,39.579,39.579,32.527,0.000,1.343,26.851
3136,171.556,160.490,160.153,164.182,177.896,173.559,189.783,196.939,173.898,170.216,...,17.260,9.341,6.412,12.481,38.236,38.236,31.184,1.343,0.000,25.508


In [6]:
async def get_model(region, towns, adj_mx, region_id, local_crs):
    try:
        region_model = Region(
            region = region.to_crs(local_crs),
            towns=towns.to_crs(local_crs),
            accessibility_matrix=adj_mx
            )
        return region_model
    except Exception as e:
        raise RuntimeError(f"Error calculating the matrix for region {region_id}: {str(e)}")

In [7]:
model = await get_model(region, towns, adj_mx, region_id, local_crs)

In [8]:
model.get_towns_gdf()

,id,name,population,level,geometry
207,207,Болото,10,Малое сельское поселение,POINT (543142.343 6580637.216)
208,208,Большой Остров,68,Малое сельское поселение,POINT (544532.933 6593227.683)
209,209,Бор,1734,Большое сельское поселение,POINT (544895.406 6593052.346)
210,210,Бороватое,10,Малое сельское поселение,POINT (543993.115 6589581.591)
211,211,Бочево,10,Малое сельское поселение,POINT (538540.114 6576793.156)
...,...,...,...,...,...
3133,3133,Апраксин Бор,313,Среднее сельское поселение,POINT (399059.733 6560340.947)
3134,3134,Александровка,313,Среднее сельское поселение,POINT (403967.163 6562086.557)
3135,3135,Большая Горка,313,Среднее сельское поселение,POINT (413116.733 6573792.753)
3136,3136,Дроздово,5,Малое сельское поселение,POINT (412458.024 6574810.78)
